In [25]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail/cnn_dailymail/validation.csv
/kaggle/input/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail/cnn_dailymail/train.csv
/kaggle/input/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail/cnn_dailymail/test.csv


In [26]:
# ── 0. Imports ───────────────────────────────────────────────
import os, math, time, random, pickle
import numpy as np
import pandas as pd
 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
 
from collections import Counter
 
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [27]:
# ── 1. Config ────────────────────────────────────────────────
class CFG:
    # Paths (Kaggle input layout)
    DATA_DIR        = "/kaggle/input/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail/cnn_dailymail"
    OUTPUT_DIR      = "/kaggle/working"
 
    # Dataset truncation
    TRAIN_SAMPLES   = 20_000        # articles to keep for training
    VAL_SAMPLES     = 2_000
 
    # Text length (in tokens)
    MAX_SRC_LEN     = 150           # article  (truncated)
    MAX_TGT_LEN     = 40            # summary  (truncated)
 
    # Vocabulary
    VOCAB_SIZE      = 30_000        # most-common words
    MIN_FREQ        = 2
 
    # Model
    EMBED_DIM       = 256
    HIDDEN_DIM      = 512
    ENC_LAYERS      = 2
    DEC_LAYERS      = 2
    DROPOUT         = 0.3
 
    # Training
    BATCH_SIZE      = 64
    EPOCHS          = 15
    LR              = 1e-3
    CLIP            = 1.0           # gradient clipping
    TEACHER_FORCING = 0.5           # probability of using ground-truth token
 
    # Special tokens
    PAD_TOKEN = "<pad>"
    SOS_TOKEN = "<sos>"
    EOS_TOKEN = "<eos>"
    UNK_TOKEN = "<unk>"

In [28]:
# ── 2. Vocabulary ────────────────────────────────────────────
class Vocabulary:
    def __init__(self):
        self.word2idx = {}
        self.idx2word = {}
        self.freq     = Counter()
        # reserve index 0 for padding
        for tok in [CFG.PAD_TOKEN, CFG.SOS_TOKEN, CFG.EOS_TOKEN, CFG.UNK_TOKEN]:
            self._add(tok)
 
    def _add(self, word):
        if word not in self.word2idx:
            idx = len(self.word2idx)
            self.word2idx[word] = idx
            self.idx2word[idx]  = word
 
    def build(self, sentences, max_size, min_freq):
        for sent in sentences:
            self.freq.update(sent.lower().split())
        for word, cnt in self.freq.most_common(max_size):
            if cnt >= min_freq:
                self._add(word)
        print(f"Vocabulary size: {len(self.word2idx)}")
 
    def encode(self, sentence, max_len):
        tokens = sentence.lower().split()[:max_len]
        return [self.word2idx.get(t, self.word2idx[CFG.UNK_TOKEN]) for t in tokens]
 
    def decode(self, indices):
        words = []
        for i in indices:
            w = self.idx2word.get(i, CFG.UNK_TOKEN)
            if w == CFG.EOS_TOKEN:
                break
            if w not in (CFG.PAD_TOKEN, CFG.SOS_TOKEN):
                words.append(w)
        return " ".join(words)
 
    def __len__(self):
        return len(self.word2idx)
 
 

In [29]:
# ── 3. Dataset ───────────────────────────────────────────────
class SummarizationDataset(Dataset):
    def __init__(self, df, vocab, src_max, tgt_max):
        self.vocab   = vocab
        self.src_max = src_max
        self.tgt_max = tgt_max
        self.data    = df[["article", "highlights"]].dropna().reset_index(drop=True)
 
    def __len__(self):
        return len(self.data)
 
    def __getitem__(self, idx):
        src_raw = self.data.loc[idx, "article"]
        tgt_raw = self.data.loc[idx, "highlights"]
 
        src_ids = self.vocab.encode(src_raw, self.src_max)
        tgt_ids = (
            [self.vocab.word2idx[CFG.SOS_TOKEN]]
            + self.vocab.encode(tgt_raw, self.tgt_max)
            + [self.vocab.word2idx[CFG.EOS_TOKEN]]
        )
        return torch.tensor(src_ids, dtype=torch.long), \
               torch.tensor(tgt_ids, dtype=torch.long)
 
 
def collate_fn(batch):
    """Pad sequences in a batch to the same length."""
    pad_id = 0  # CFG.PAD_TOKEN index
    srcs, tgts = zip(*batch)
 
    src_lens = [len(s) for s in srcs]
    tgt_lens = [len(t) for t in tgts]
 
    src_pad = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=pad_id)
    tgt_pad = nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=pad_id)
 
    return src_pad, tgt_pad, torch.tensor(src_lens), torch.tensor(tgt_lens)

In [30]:
# ── 4. Model ─────────────────────────────────────────────────
 
class Encoder(nn.Module):
    """Multi-layer bidirectional LSTM encoder."""
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, n_layers,
            batch_first=True, dropout=dropout if n_layers > 1 else 0,
            bidirectional=True
        )
        # Project bidirectional hidden/cell → decoder hidden size
        self.fc_h = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_c = nn.Linear(hidden_dim * 2, hidden_dim)
        self.dropout = nn.Dropout(dropout)
 
    def forward(self, src, src_lens):
        # src : [B, src_len]
        embedded = self.dropout(self.embedding(src))          # [B, src_len, E]
 
        packed = pack_padded_sequence(
            embedded, src_lens.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (hidden, cell) = self.lstm(packed)
        # hidden : [n_layers*2, B, H]  →  reshape to [n_layers, B, 2*H]
        n_layers = hidden.shape[0] // 2
        hidden = hidden.view(n_layers, 2, -1, hidden.shape[-1])   # [L, 2, B, H]
        cell   = cell.view(n_layers, 2, -1, cell.shape[-1])
 
        hidden = torch.cat([hidden[:, 0], hidden[:, 1]], dim=-1)  # [L, B, 2H]
        cell   = torch.cat([cell[:, 0],   cell[:, 1]],   dim=-1)
 
        hidden = torch.tanh(self.fc_h(hidden))   # [L, B, H]
        cell   = torch.tanh(self.fc_c(cell))     # [L, B, H]
        return hidden, cell
 
 
class Decoder(nn.Module):
    """Multi-layer unidirectional LSTM decoder (no attention)."""
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, n_layers,
            batch_first=True, dropout=dropout if n_layers > 1 else 0
        )
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)
 
    def forward_step(self, token, hidden, cell):
        # token : [B]
        embedded = self.dropout(self.embedding(token.unsqueeze(1)))  # [B, 1, E]
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell)) # output: [B,1,H]
        pred = self.fc_out(output.squeeze(1))                        # [B, vocab]
        return pred, hidden, cell
 
 
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, tgt_vocab_size):
        super().__init__()
        self.encoder       = encoder
        self.decoder       = decoder
        self.tgt_vocab_size = tgt_vocab_size
        self.sos_idx = 1    # fixed by Vocabulary construction order
        self.eos_idx = 2
 
    def forward(self, src, src_lens, tgt, teacher_forcing_ratio=0.5):
        """
        src  : [B, src_len]
        tgt  : [B, tgt_len]   (includes <sos> at position 0)
        """
        B, tgt_len = tgt.shape
        outputs = torch.zeros(B, tgt_len, self.tgt_vocab_size).to(src.device)
 
        hidden, cell = self.encoder(src, src_lens)
 
        # First decoder input = <sos>
        token = tgt[:, 0]
 
        for t in range(1, tgt_len):
            pred, hidden, cell = self.decoder.forward_step(token, hidden, cell)
            outputs[:, t] = pred
            use_teacher = random.random() < teacher_forcing_ratio
            token = tgt[:, t] if use_teacher else pred.argmax(dim=-1)
 
        return outputs
 
    @torch.no_grad()
    def generate(self, src, src_lens, max_len=50):
        """Greedy decoding for a single batch."""
        self.eval()
        hidden, cell = self.encoder(src, src_lens)
        token = torch.full((src.shape[0],), self.sos_idx,
                           dtype=torch.long, device=src.device)
        generated = []
        for _ in range(max_len):
            pred, hidden, cell = self.decoder.forward_step(token, hidden, cell)
            token = pred.argmax(dim=-1)
            generated.append(token.cpu().numpy())
            if (token == self.eos_idx).all():
                break
        # generated : list of [B] → transpose → [B, steps]
        return np.array(generated).T
 

In [31]:
# ── 5. Training helpers ──────────────────────────────────────
 
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
 
def train_epoch(model, loader, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    for src, tgt, src_lens, _ in loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
 
        optimizer.zero_grad()
        output = model(src, src_lens, tgt, CFG.TEACHER_FORCING)
        # output : [B, tgt_len, vocab]  — skip <sos> at index 0
        output_flat = output[:, 1:].reshape(-1, output.shape[-1])
        tgt_flat    = tgt[:, 1:].reshape(-1)
        loss = criterion(output_flat, tgt_flat)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)
 
 
def eval_epoch(model, loader, criterion):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for src, tgt, src_lens, _ in loader:
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)
            output = model(src, src_lens, tgt, teacher_forcing_ratio=0.0)
            output_flat = output[:, 1:].reshape(-1, output.shape[-1])
            tgt_flat    = tgt[:, 1:].reshape(-1)
            loss = criterion(output_flat, tgt_flat)
            epoch_loss += loss.item()
    return epoch_loss / len(loader)
 

In [ ]:
# ── 6. Main ──────────────────────────────────────────────────
 
def main():
    # 6-a. Load & truncate data
    print("Loading data...")
    train_df = pd.read_csv(os.path.join(CFG.DATA_DIR, "train.csv")).sample(
        CFG.TRAIN_SAMPLES, random_state=SEED).reset_index(drop=True)
    val_df = pd.read_csv(os.path.join(CFG.DATA_DIR, "validation.csv")).sample(
        CFG.VAL_SAMPLES, random_state=SEED).reset_index(drop=True)
 
    print(f"Train size : {len(train_df)} | Val size : {len(val_df)}")
 
    # 6-b. Build vocabulary on training set only
    print("Building vocabulary...")
    vocab = Vocabulary()
    vocab.build(
        list(train_df["article"]) + list(train_df["highlights"]),
        max_size=CFG.VOCAB_SIZE,
        min_freq=CFG.MIN_FREQ,
    )
 
    # 6-c. Datasets & loaders
    train_ds = SummarizationDataset(train_df, vocab, CFG.MAX_SRC_LEN, CFG.MAX_TGT_LEN)
    val_ds   = SummarizationDataset(val_df,   vocab, CFG.MAX_SRC_LEN, CFG.MAX_TGT_LEN)
 
    train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE,
                              shuffle=True,  collate_fn=collate_fn, num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=CFG.BATCH_SIZE,
                              shuffle=False, collate_fn=collate_fn, num_workers=2)
 
    # 6-d. Build model
    encoder = Encoder(len(vocab), CFG.EMBED_DIM, CFG.HIDDEN_DIM,
                      CFG.ENC_LAYERS, CFG.DROPOUT)
    decoder = Decoder(len(vocab), CFG.EMBED_DIM, CFG.HIDDEN_DIM,
                      CFG.DEC_LAYERS, CFG.DROPOUT)
    model = Seq2Seq(encoder, decoder, len(vocab)).to(DEVICE)
 
    print(f"Model has {count_parameters(model):,} trainable parameters")
 
    # 6-e. Optimizer & loss  (ignore padding index=0)
    optimizer = optim.Adam(model.parameters(), lr=CFG.LR)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=2)
    criterion = nn.CrossEntropyLoss(ignore_index=0)
 
    # 6-f. Training loop
    best_val_loss = float("inf")
    history = {"train_loss": [], "val_loss": []}
 
    for epoch in range(1, CFG.EPOCHS + 1):
        t0 = time.time()
        train_loss = train_epoch(model, train_loader, optimizer, criterion, CFG.CLIP)
        val_loss   = eval_epoch(model, val_loader, criterion)
        prev_lr = optimizer.param_groups[0]["lr"]
        scheduler.step(val_loss)
        new_lr = optimizer.param_groups[0]["lr"]
        if new_lr != prev_lr:
            print(f"  ↓ LR reduced: {prev_lr:.2e} → {new_lr:.2e}")
 
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
 
        elapsed = time.time() - t0
        print(
            f"Epoch {epoch:02d}/{CFG.EPOCHS} | "
            f"Time: {elapsed:.0f}s | "
            f"Train Loss: {train_loss:.4f} (PPL {math.exp(train_loss):.2f}) | "
            f"Val Loss: {val_loss:.4f} (PPL {math.exp(val_loss):.2f})"
        )
 
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(),
                       os.path.join(CFG.OUTPUT_DIR, "best_model.pt"))
            print("  ✓ Best model saved.")
 
    print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")
 
    # 6-g. Save vocab + full checkpoint (for GitHub / reuse)
    vocab_path = os.path.join(CFG.OUTPUT_DIR, "vocab.pkl")
    with open(vocab_path, "wb") as f:
        pickle.dump(vocab, f)
 
    checkpoint = {
        "model_state_dict" : model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "vocab_size"       : len(vocab),
        "embed_dim"        : CFG.EMBED_DIM,
        "hidden_dim"       : CFG.HIDDEN_DIM,
        "enc_layers"       : CFG.ENC_LAYERS,
        "dec_layers"       : CFG.DEC_LAYERS,
        "dropout"          : CFG.DROPOUT,
        "history"          : history,
        "best_val_loss"    : best_val_loss,
    }
    ckpt_path = os.path.join(CFG.OUTPUT_DIR, "seq2seq_checkpoint.pt")
    torch.save(checkpoint, ckpt_path)
    print(f"Full checkpoint saved → {ckpt_path}")
    print(f"Vocabulary saved      → {vocab_path}")
 
    # 6-h. Quick qualitative check
    print("\n── Sample Predictions ──────────────────────────────")
    model.load_state_dict(torch.load(
        os.path.join(CFG.OUTPUT_DIR, "best_model.pt"), map_location=DEVICE))
    model.eval()
 
    samples = val_df.sample(3, random_state=1).reset_index(drop=True)
    for i, row in samples.iterrows():
        src_ids = torch.tensor(
            vocab.encode(row["article"], CFG.MAX_SRC_LEN), dtype=torch.long
        ).unsqueeze(0).to(DEVICE)
        src_len = torch.tensor([src_ids.shape[1]])
 
        pred_ids = model.generate(src_ids, src_len, max_len=CFG.MAX_TGT_LEN + 10)
        pred_text = vocab.decode(pred_ids[0].tolist())
 
        print(f"\n[{i+1}] Article snippet : {row['article'][:200]} ...")
        print(f"    Reference       : {row['highlights']}")
        print(f"    Prediction      : {pred_text}")
 
    print("\nDone. Files in /kaggle/working:")
    for f in os.listdir(CFG.OUTPUT_DIR):
        fpath = os.path.join(CFG.OUTPUT_DIR, f)
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"  {f}  ({size_mb:.1f} MB)")
 
 
if __name__ == "__main__":
    main()
 

Loading data...
Train size : 20000 | Val size : 2000
Building vocabulary...
Vocabulary size: 30004
Model has 44,935,476 trainable parameters
Epoch 01/15 | Time: 224s | Train Loss: 7.1969 (PPL 1335.23) | Val Loss: 6.9395 (PPL 1032.20)
  ✓ Best model saved.
Epoch 02/15 | Time: 225s | Train Loss: 6.8978 (PPL 990.09) | Val Loss: 6.8795 (PPL 972.12)
  ✓ Best model saved.
Epoch 03/15 | Time: 225s | Train Loss: 6.7326 (PPL 839.32) | Val Loss: 6.9203 (PPL 1012.62)
Epoch 04/15 | Time: 225s | Train Loss: 6.6279 (PPL 755.89) | Val Loss: 6.8581 (PPL 951.57)
  ✓ Best model saved.
Epoch 05/15 | Time: 225s | Train Loss: 6.5513 (PPL 700.18) | Val Loss: 6.8724 (PPL 965.23)
Epoch 06/15 | Time: 226s | Train Loss: 6.4980 (PPL 663.81) | Val Loss: 6.8611 (PPL 954.43)
  ↓ LR reduced: 1.00e-03 → 5.00e-04
Epoch 07/15 | Time: 226s | Train Loss: 6.4396 (PPL 626.18) | Val Loss: 6.8634 (PPL 956.58)
Epoch 08/15 | Time: 225s | Train Loss: 6.3640 (PPL 580.56) | Val Loss: 6.8671 (PPL 960.18)
Epoch 09/15 | Time: 225s |

In [33]:
from IPython.display import FileLink

FileLink('/kaggle/working/seq2seq_checkpoint.pt')

/kaggle/working/seq2seq_checkpoint.pt